# Checking out the mixed muon database

Udforsker den blandede database (1000 MC + 1000 BS muon-events):
- **Path**: `mixed_1000_mc_1000_bs_muons.db`

Indeholder: `is_mc`, `source`, `original_event_no`, alle MC truth-kolonner, samt PID-predictions (`pid_noise_pred`, `pid_muon_pred`, `pid_neutrino_pred`).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)

DB_PATH = '/groups/icecube/holgerkc/Thesis_Analysis/MC_vs_BS_analysis/Data/results/mixed_5000_mc_5000_bs_muons.db'

con = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", con
)['name'].tolist()
print('Tables:', tables)

---
## 1. Schema — kolonner i `truth`

In [ ]:
# truth columns
truth_cols = pd.read_sql_query('SELECT * FROM truth LIMIT 1;', con)
print('truth columns:')
print(truth_cols.columns.tolist())

---
## 2. Event counts — MC vs BS

In [ ]:
# Total rows per table
for t in tables:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t};', con).iloc[0]['n']
    print(f'{t}: {int(n)} rows')

# MC vs BS split i truth
print()
split = pd.read_sql_query(
    "SELECT source, is_mc, COUNT(*) AS n_events FROM truth GROUP BY source, is_mc ORDER BY is_mc;",
    con,
)
n_mc = int(split.loc[split['is_mc'] == 1, 'n_events'].iloc[0])
n_bs = int(split.loc[split['is_mc'] == 0, 'n_events'].iloc[0])
print('MC vs BS fordeling:')
print(split)

---
## 3. Sample events fra `truth`

In [ ]:
# 10 MC events — alle truth-kolonner, transponeret
truth_mc = pd.read_sql_query(
    "SELECT * FROM truth WHERE is_mc = 1 ORDER BY event_no LIMIT 10;", con
)
print('MC events (transponeret):')
print(truth_mc.T)

In [ ]:
# 10 BS events — transponeret (MC truth-kolonner er NULL, PID-predictions er udfyldt)
truth_bs = pd.read_sql_query(
    "SELECT * FROM truth WHERE is_mc = 0 ORDER BY event_no LIMIT 10;", con
)
print('BS events (transponeret):')
print(truth_bs.T)

---
## 4. PID predictions

In [ ]:
# PID predictions for alle events
pid_df = pd.read_sql_query(
    "SELECT event_no, is_mc, source, pid, pid_noise_pred, pid_muon_pred, pid_neutrino_pred FROM truth ORDER BY event_no;",
    con,
)
print(f'Events med PID predictions: {pid_df["pid_muon_pred"].notna().sum()} / {len(pid_df)}')
print()
print('MC events (pid kendes):')
print(pid_df[pid_df['is_mc'] == 1][['event_no', 'pid', 'pid_noise_pred', 'pid_muon_pred', 'pid_neutrino_pred']].head(10).to_string(index=False))
print()
print('BS events (pid = None, kun predictions):')
print(pid_df[pid_df['is_mc'] == 0][['event_no', 'pid_noise_pred', 'pid_muon_pred', 'pid_neutrino_pred']].head(10).to_string(index=False))

---
## 5. SplitInIcePulses — sample for ét event

In [ ]:
# SplitInIcePulses kolonner
pulses_cols = pd.read_sql_query('SELECT * FROM SplitInIcePulses LIMIT 1;', con)
print('SplitInIcePulses columns:', pulses_cols.columns.tolist())

In [ ]:
# Pulses for det første MC event
sample_event_mc = int(truth_mc.iloc[0]['event_no'])
pulses_mc = pd.read_sql_query(
    f'SELECT * FROM SplitInIcePulses WHERE event_no = {sample_event_mc};', con
)
print(f'Pulses for MC event_no={sample_event_mc} ({len(pulses_mc)} pulses):')
print(pulses_mc)

In [ ]:
# Pulses for det første BS event
sample_event_bs = int(truth_bs.iloc[0]['event_no'])
pulses_bs = pd.read_sql_query(
    f'SELECT * FROM SplitInIcePulses WHERE event_no = {sample_event_bs};', con
)
print(f'Pulses for BS event_no={sample_event_bs} ({len(pulses_bs)} pulses):')
print(pulses_bs)

---
## 6. 3D visualisering af DOM-positioner

Sammenligning af ét MC event og ét BS event. Markerstørrelse = charge.

In [ ]:
def plot_event_3d(pulses: pd.DataFrame, title: str) -> go.Figure:
    charge = pulses['charge'].to_numpy()
    sizes = np.clip(charge * 5, 2, 30)
    fig = go.Figure(data=[go.Scatter3d(
        x=pulses['dom_x'],
        y=pulses['dom_y'],
        z=pulses['dom_z'],
        mode='markers',
        marker=dict(
            size=sizes,
            color=charge,
            colorscale='Viridis',
            opacity=0.8,
            colorbar=dict(title='Charge'),
        ),
        text=[f'charge={c:.3f}<br>t={t:.0f}' for c, t in zip(pulses['charge'], pulses['dom_time'])],
        hoverinfo='text',
    )])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title='X [m]', yaxis_title='Y [m]', zaxis_title='Z [m]'),
        width=750, height=600,
    )
    return fig

# MC event
fig_mc = plot_event_3d(
    pulses_mc,
    f'MC event_no={sample_event_mc} — pid={int(truth_mc.iloc[0]["pid"])} — {len(pulses_mc)} pulses',
)
fig_mc.show()

In [ ]:
# BS event
fig_bs = plot_event_3d(
    pulses_bs,
    f'BS event_no={sample_event_bs} — {len(pulses_bs)} pulses',
)
fig_bs.show()

con.close()